# Drug Safety Signal Explorer

**Goal:** Analyze a defined sample of openFDA adverse-event reports, normalize the nested JSON data, and identify drug-reaction pairs with disproportionate reporting.

> Potential signals indicate disproportionate reporting within the analyzed sample. They do not establish causality.

In [7]:
# Export the final signal dataset as a CSV for Tableau
import requests
url = "https://api.fda.gov/drug/event.json"

response = requests.get(url)

print(response.status_code)

200


## Understanding the API Response

In [8]:
print(response.text)

{
  "meta": {
    "disclaimer": "Do not rely on openFDA to make decisions regarding medical care. While we make every effort to ensure that data is accurate, you should assume all results are unvalidated. We may limit or otherwise restrict your access to the API in line with our Terms of Service.",
    "terms": "https://open.fda.gov/terms/",
    "license": "https://open.fda.gov/license/",
    "last_updated": "2026-07-30",
    "results": {
      "skip": 0,
      "limit": 1,
      "total": 20692690
    }
  },
  "results": [
    {
      "safetyreportid": "5801206-7",
      "transmissiondateformat": "102",
      "transmissiondate": "20090109",
      "serious": "1",
      "seriousnessdeath": "1",
      "receivedateformat": "102",
      "receivedate": "20080707",
      "receiptdateformat": "102",
      "receiptdate": "20080625",
      "fulfillexpeditecriteria": "1",
      "companynumb": "JACAN16471",
      "primarysource": {
        "reportercountry": "CANADA",
        "qualification": "3"
 

In [9]:
data = response.json()
print(type(data))

<class 'dict'>


In [10]:
print(data.keys())

dict_keys(['meta', 'results'])


In [11]:
print(type(data["results"]))

<class 'list'>


```
So the structure is:

data
 │
 ├── meta       → dictionary
 │
 └── results    → list
                    │
                    ├── report 1 → dictionary
                    ├── report 2 → dictionary
                    ├── report 3 → dictionary
                    └── ...

In [12]:
import requests

BASE = "https://api.fda.gov/drug/event.json"

response = requests.get(
    BASE,
    params = {
        "search": 'patient.reaction.reactionmeddrapt:"headache"',
        "limit": 5
    }
)
print(response.status_code)
data = response.json() #Take the JSON returned by the API and convert it into something Python can work with like dictionary or list

print(data.keys()) #shows the top-level keys in the dictionary.
print(data["meta"]["results"]["total"]) #how many matching records exist overall
print(len(data["results"])) #how many records we actually received in this API response

200
dict_keys(['meta', 'results'])
629244
5


In [13]:
## Understanding Nested Report Structure
report = {
    "safetyreportid": "12345",
    "patient": {
        "drug": [
            {"medicinalproduct": "ASPIRIN"},
            {"medicinalproduct": "IBUPROFEN"}
        ],
      "reaction": [
          {"reactionmeddrapt": "HEADACHE"},
          {"reactionmeddrapt": "NAUSEA"}
      ]
    }
}

In [14]:
# Explore different ways to flatten nested JSON data
import pandas as pd

df = pd.json_normalize(report)

df

,safetyreportid,patient.drug,patient.reaction
0,12345,"[{'medicinalproduct': 'ASPIRIN'}, {'medicinalp...","[{'reactionmeddrapt': 'HEADACHE'}, {'reactionm..."


In [18]:
df_drugs = df.explode("patient.drug").reset_index(drop=True) #json_normalize() helps flatten nested dictionaries. explode() helps turn lists inside cells into separate rows.
df_drugs
#explode() creates new rows but preserves the original index.

,safetyreportid,patient.drug,patient.reaction
0,12345,{'medicinalproduct': 'ASPIRIN'},"[{'reactionmeddrapt': 'HEADACHE'}, {'reactionm..."
1,12345,{'medicinalproduct': 'IBUPROFEN'},"[{'reactionmeddrapt': 'HEADACHE'}, {'reactionm..."


In [19]:
df_reactions = df.explode("patient.reaction").reset_index(drop=True)
df_reactions

,safetyreportid,patient.drug,patient.reaction
0,12345,"[{'medicinalproduct': 'ASPIRIN'}, {'medicinalp...",{'reactionmeddrapt': 'HEADACHE'}
1,12345,"[{'medicinalproduct': 'ASPIRIN'}, {'medicinalp...",{'reactionmeddrapt': 'NAUSEA'}


In [20]:
df_drugs["drug"] = df_drugs["patient.drug"].apply(
    lambda x: x["medicinalproduct"]
)
df_drugs

,safetyreportid,patient.drug,patient.reaction,drug
0,12345,{'medicinalproduct': 'ASPIRIN'},"[{'reactionmeddrapt': 'HEADACHE'}, {'reactionm...",ASPIRIN
1,12345,{'medicinalproduct': 'IBUPROFEN'},"[{'reactionmeddrapt': 'HEADACHE'}, {'reactionm...",IBUPROFEN


In [21]:
df_reactions["reactions"] = df_reactions["patient.reaction"].apply(
    lambda x:x["reactionmeddrapt"]
)
df_reactions

,safetyreportid,patient.drug,patient.reaction,reactions
0,12345,"[{'medicinalproduct': 'ASPIRIN'}, {'medicinalp...",{'reactionmeddrapt': 'HEADACHE'},HEADACHE
1,12345,"[{'medicinalproduct': 'ASPIRIN'}, {'medicinalp...",{'reactionmeddrapt': 'NAUSEA'},NAUSEA


In [22]:
# Extract drug information while preserving the report ID
drugs_rows = []

for drug in report["patient"]["drug"]:
  drugs_rows.append({
      "reportid": report["safetyreportid"],
      "drug": drug["medicinalproduct"]
  })
drugs_df = pd.DataFrame(drugs_rows)
drugs_df

,reportid,drug
0,12345,ASPIRIN
1,12345,IBUPROFEN


In [23]:
reaction_rows = []

for reaction in report["patient"]["reaction"]:
  reaction_rows.append({
      "reportid": report["safetyreportid"],
      "reaction": reaction["reactionmeddrapt"]
  })
reactions_df = pd.DataFrame(reaction_rows)
reactions_df


,reportid,reaction
0,12345,HEADACHE
1,12345,NAUSEA


In [24]:
# Explore API search parameters and pagination
import requests

BASE = "https://api.fda.gov/drug/event.json"

params1 = {
    "search": 'patient.reaction.reactionmeddrapt:headache',
    "limit": 5,
    "skip": 0
}

response1 = requests.get(BASE, params=params1)
data1 = response1.json()

In [25]:
params2 = {
    "search": 'patient.reaction.reactionmeddrapt:headache',
    "limit": 5,
    "skip": 5
}

response2 = requests.get(BASE, params=params2)
data2 = response2.json()

In [26]:
data1["results"][0] #Give me the first report from the first batch.
data2["results"][0]#Give me the first report from the second batch.

{'safetyreportversion': '2',
 'safetyreportid': '10003390',
 'primarysourcecountry': 'US',
 'occurcountry': 'US',
 'transmissiondateformat': '102',
 'transmissiondate': '20151125',
 'reporttype': '1',
 'serious': '2',
 'receivedateformat': '102',
 'receivedate': '20140312',
 'receiptdateformat': '102',
 'receiptdate': '20150812',
 'fulfillexpeditecriteria': '2',
 'companynumb': 'US-GILEAD-2012-0062268',
 'duplicate': '1',
 'reportduplicate': {'duplicatesource': 'GILEAD',
  'duplicatenumb': 'US-GILEAD-2012-0062268'},
 'primarysource': {'reportercountry': 'US', 'qualification': '5'},
 'sender': {'sendertype': '2', 'senderorganization': 'FDA-Public Use'},
 'receiver': {'receivertype': '6', 'receiverorganization': 'FDA'},
 'patient': {'patientonsetage': '44',
  'patientonsetageunit': '801',
  'patientagegroup': '5',
  'patientsex': '2',
  'reaction': [{'reactionmeddraversionpt': '18.1',
    'reactionmeddrapt': 'Headache',
    'reactionoutcome': '6'},
   {'reactionmeddraversionpt': '18.1',


## Collect a defined 10,000-report sample

In [28]:
# Fetch reports in batches using the skip and limit parameters
# Add retry handling for failed API requests
import requests
import time

BASE = "https://api.fda.gov/drug/event.json"

def fetch_batch(search, limit=100, skip=0, retries=3):
  params = {
      "search": search,
      "limit": limit,
      "skip": skip

  }

  for attempt in range(retries):
    response = requests.get(BASE, params=params)

    if response.status_code == 200:
      return response.json()

    print(f"attempt : {attempt + 1}, failed: {response.status_code}")

    time.sleep(2)
  return None

In [29]:
data = fetch_batch(
    'patient.reaction.reactionmeddrapt:headache',
    limit=100,
    skip=0
)

In [30]:
len(data["results"])


100

In [31]:
# Cache collected reports locally to avoid repeated API requests
import json
import os

CACHE_FILE = "openfda_cache.json"

In [33]:
with open(CACHE_FILE, "w") as f:
  json.dump(all_reports, f)


In [34]:
with open(CACHE_FILE, "r") as f:
  all_reports = json.load(f)
print(len(all_reports))

500


In [35]:
import pandas as pd

df = pd.json_normalize(all_reports)

print(df.shape)
print(df.columns.to_list())

(500, 39)
['safetyreportversion', 'safetyreportid', 'primarysourcecountry', 'transmissiondateformat', 'transmissiondate', 'reporttype', 'serious', 'seriousnessdisabling', 'receivedateformat', 'receivedate', 'receiptdateformat', 'receiptdate', 'fulfillexpeditecriteria', 'companynumb', 'duplicate', 'reportduplicate.duplicatesource', 'reportduplicate.duplicatenumb', 'primarysource.reportercountry', 'primarysource.qualification', 'sender.sendertype', 'sender.senderorganization', 'receiver.receivertype', 'receiver.receiverorganization', 'patient.patientonsetage', 'patient.patientonsetageunit', 'patient.patientsex', 'patient.reaction', 'patient.drug', 'occurcountry', 'patient.patientagegroup', 'seriousnesshospitalization', 'patient.summary.narrativeincludeclinical', 'seriousnesslifethreatening', 'seriousnessother', 'patient.patientweight', 'seriousnessdeath', 'primarysource.literaturereference', 'authoritynumb', 'seriousnesscongenitalanomali']


## Initial Data Inspection

In [36]:
# Check missing values across the dataset
df.isna().sum().sort_values(ascending=False)

,0
seriousnesscongenitalanomali,499
authoritynumb,499
primarysource.literaturereference,499
seriousnesslifethreatening,493
seriousnessdeath,493
seriousnessdisabling,486
seriousnesshospitalization,427
patient.summary.narrativeincludeclinical,416
patient.patientweight,404
seriousnessother,380


In [37]:
# Check for duplicate reports using the report identifier
print("Total rows:", len(df))
df["safetyreportid"].duplicated().sum()

Total rows: 500


np.int64(0)

In [38]:
df["duplicate"].value_counts(dropna=False)

,count
duplicate,
1,491
NaN,9


In [39]:
# Reload and continue working with the cleaned dataset
df = pd.json_normalize(all_reports)

print("Total rows:", len(df))
print("Columns:", len(df.columns))

Total rows: 500
Columns: 39


In [40]:
# Convert date fields into a consistent format
df["receivedate"] = pd.to_datetime(df["receivedate"].astype(str), format="%Y%m%d", errors="coerce")



In [41]:
df["receivedate"].head()

,receivedate
0,2014-03-06
1,2014-03-12
2,2014-03-12
3,2014-03-12
4,2014-03-12


In [42]:
df["receivedate"].dtype

dtype('<M8[ns]')

In [43]:
df["receivedate"].isna().sum()

np.int64(0)

In [45]:
# Inspect and convert patient age information to numeric values
df["patient.patientonsetageunit"].value_counts(dropna=False)

,count
patient.patientonsetageunit,
801,389
NaN,111


In [46]:
df["patient.patientonsetage"].describe()

,patient.patientonsetage
count,389
unique,71
top,53
freq,14


In [47]:
df["patient.patientonsetage"] = pd.to_numeric(df["patient.patientonsetage"], errors="coerce")


In [48]:
df["patient.patientonsetage"].describe()

,patient.patientonsetage
count,389.000000
mean,55.123393
std,15.616545
min,5.000000
25%,46.000000
50%,56.000000
75%,67.000000
max,88.000000


In [49]:
# Inspect sex values and map coded values to readable categories
df["patient.patientsex"].value_counts(dropna=False)

,count
patient.patientsex,
2,361
1,128
0,8
NaN,3


In [50]:
sex_map = {
    "1": "Male",
    "2": "Female",
    "0": "Unknown"
}

df["sex"] = df["patient.patientsex"].replace(sex_map)

df["sex"].value_counts(dropna=False)

,count
sex,
Female,361
Male,128
Unknown,8
NaN,3


In [51]:
# Inspect country information in the reports
df["primarysourcecountry"].value_counts(dropna=False).head(15)

,count
primarysourcecountry,
US,420
CA,21
GB,14
BR,5
FR,5
DE,5
CM,4
CN,3
AU,2


In [52]:
df["occurcountry"].value_counts(dropna=False).head(15)

,count
occurcountry,
US,403
NaN,24
CA,21
GB,13
BR,5
FR,5
DE,4
CN,3
AU,2


## Extracting Drugs and Reactions

In [55]:
df["patient.drug"].iloc[0]

[{'drugcharacterization': '1',
  'medicinalproduct': 'BONIVA',
  'drugbatchnumb': 'H6200HO3',
  'drugauthorizationnumb': '021858',
  'drugstructuredosagenumb': '3',
  'drugstructuredosageunit': '003',
  'drugdosagetext': '3 MG, 1 IN 3 M, INTRAVENOUS (NOT OTHERWISE SPECIFIED)',
  'drugadministrationroute': '042',
  'drugindication': 'OSTEOPOROSIS',
  'drugstartdateformat': '102',
  'drugstartdate': '20130913'}]

In [57]:
df["patient.reaction"].iloc[0]

[{'reactionmeddraversionpt': '17.0', 'reactionmeddrapt': 'Vomiting'},
 {'reactionmeddraversionpt': '17.0', 'reactionmeddrapt': 'Diarrhoea'},
 {'reactionmeddraversionpt': '17.0', 'reactionmeddrapt': 'Arthralgia'},
 {'reactionmeddraversionpt': '17.0', 'reactionmeddrapt': 'Headache'}]

In [58]:
# Extract one row per drug per report
drug_rows = []

for report in all_reports:
  report_id = report.get("safetyreportid")

  for drug in report.get("patient", {}).get("drug", []):
    drug_rows.append(
        {
            "report_id" : report_id,
            "drug" : drug.get("medicinalproduct")
        }

    )
drugs_df = pd.DataFrame(drug_rows)

print(drugs_df.shape)
drugs_df.head()

(1918, 2)


,report_id,drug
0,10003300,BONIVA
1,10003320,TYVASO
2,10003320,LOSARTAN.
3,10003320,LETAIRIS
4,10003320,LOSARTAN.


In [59]:
# Extract one row per reaction per report
reaction_rows = []

for report in all_reports:
    report_id = report.get("safetyreportid")

    for reaction in report.get("patient", {}).get("reaction", []):
        reaction_rows.append({
            "report_id": report_id,
            "reaction": reaction.get("reactionmeddrapt")
        })

reactions_df = pd.DataFrame(reaction_rows)

print(reactions_df.shape)
reactions_df.head()

(3396, 2)


,report_id,reaction
0,10003300,Vomiting
1,10003300,Diarrhoea
2,10003300,Arthralgia
3,10003300,Headache
4,10003320,Headache


In [60]:
# Check for duplicate drug/reaction entries within individual reports
drugs_df.duplicated(subset=["report_id", "drug"]).sum()

np.int64(334)

In [61]:
reactions_df.duplicated(subset=["report_id", "reaction"]).sum()

np.int64(39)

In [62]:
# Remove duplicate drug/reaction entries within the same report
drugs_df = drugs_df.drop_duplicates(subset=["report_id", "drug"]).reset_index(drop=True)

reactions_df = reactions_df.drop_duplicates(subset=["report_id", "reaction"]).reset_index(drop=True)

In [63]:
# Standardize drug and reaction names for consistent analysis
drugs_df["drug"] = drugs_df["drug"].str.upper().str.strip()

In [64]:
reactions_df["reaction"] = reactions_df["reaction"].str.upper().str.strip()

## Final Dataset Collection — 1,000 Reports

In [96]:
# Collect the final 10,000 reports using pagination, retry handling, and delays

import time
import requests

BASE = "https://api.fda.gov/drug/event.json"

all_reports = []

limit = 100
total_reports = 10000

for skip in range(0, total_reports, limit):

    for attempt in range(3):

        try:
            response = requests.get(
                BASE,
                params={
                    "limit": limit,
                    "skip": skip
                },
                timeout=30
            )

            if response.status_code == 200:
                data = response.json()
                all_reports.extend(data["results"])
                break

            print(f"Request failed at skip={skip}, attempt={attempt + 1}")

        except requests.RequestException as e:
            print(f"Request error at skip={skip}, attempt={attempt + 1}: {e}")

        time.sleep(2)

    time.sleep(0.2)

print("Total reports collected:", len(all_reports))

Total reports collected: 10000


In [97]:
len(all_reports)

10000

In [98]:
# Rebuild drug and reaction tables from the final dataset
import pandas as pd
drug_rows = []

for report in all_reports:
    report_id = report.get("safetyreportid")

    for drug in report.get("patient", {}).get("drug", []):
        drug_rows.append({
            "report_id": report_id,
            "drug": drug.get("medicinalproduct")
        })

drugs_df = pd.DataFrame(drug_rows)

print(drugs_df.shape)
drugs_df.head()

(32226, 2)


,report_id,drug
0,5801206-7,DURAGESIC-100
1,10003300,BONIVA
2,10003301,IBUPROFEN
3,10003302,LYRICA
4,10003304,DOXYCYCLINE HYCLATE


In [99]:
reaction_rows = []

for report in all_reports:
    report_id = report.get("safetyreportid")

    for reaction in report.get("patient", {}).get("reaction", []):
        reaction_rows.append({
            "report_id": report_id,
            "reaction": reaction.get("reactionmeddrapt")
        })

reactions_df = pd.DataFrame(reaction_rows)

print(reactions_df.shape)
reactions_df.head()

(26687, 2)


,report_id,reaction
0,5801206-7,DRUG ADMINISTRATION ERROR
1,5801206-7,OVERDOSE
2,10003300,Vomiting
3,10003300,Diarrhoea
4,10003300,Arthralgia


In [100]:
# Clean and deduplicate the final drug and reaction tables
drugs_df["drug"] = (drugs_df["drug"].str.strip().str.upper())

reactions_df["reaction"] = (reactions_df["reaction"].str.strip().str.upper())

In [101]:
drugs_df = drugs_df.drop_duplicates(subset=["report_id", "drug"]).reset_index(drop=True)

reactions_df = reactions_df.drop_duplicates(subset=["report_id", "reaction"]).reset_index(drop=True)

In [102]:
# Validate the final extracted data
print("Reports:", len(all_reports))
print("Drug rows:", len(drugs_df))
print("Reaction rows:", len(reactions_df))

Reports: 10000
Drug rows: 27521
Reaction rows: 26451


In [126]:
print(drugs_df["drug"].nunique())
print(reactions_df["reaction"].nunique())

5273
2954


## Creating Drug–Reaction Pairs

In [103]:
# Combine drugs and reactions that occur within the same report
pairs_df = drugs_df.merge(reactions_df, on="report_id", how="inner")

In [104]:
# Validate the generated drug-reaction combinations
pair_counts = (pairs_df.groupby(["drug", "reaction"]).size().reset_index(name="report_count"))

In [105]:
# Count how frequently each drug-reaction pair occurs
pair_counts = pair_counts.sort_values("report_count", ascending=False)
pair_counts.head(20)

,drug,reaction,report_count
36813,LIPITOR,TYPE 2 DIABETES MELLITUS,477
34876,LETAIRIS,DYSPNOEA,264
25927,"FORANE (ISOFLURANE, USP)",HYPERTHERMIA MALIGNANT,170
32129,JAKAFI,FATIGUE,143
58556,SUCCINYLCHOLINE,HYPERTHERMIA MALIGNANT,140
32326,JAKAFI,PLATELET COUNT DECREASED,133
32164,JAKAFI,HAEMOGLOBIN DECREASED,129
34948,LETAIRIS,HEADACHE,126
34857,LETAIRIS,DIZZINESS,120
31977,JAKAFI,ANAEMIA,117


## Reporting Odds Ratio (ROR)

ROR is used to measure whether a drug-reaction pair is reported disproportionately compared with other drug-reaction combinations.

In [106]:
drug = "LETAIRIS"
reaction = "DYSPNOEA"

drug_reports = set(
    drugs_df.loc[
        drugs_df["drug"] == drug,
        "report_id"
    ]
)

reaction_reports = set(
    reactions_df.loc[
        reactions_df["reaction"] == reaction,
        "report_id"
    ]
)

all_report_ids = set(
    report.get("safetyreportid")
    for report in all_reports
)

A = len(drug_reports & reaction_reports)
B = len(drug_reports - reaction_reports)
C = len(reaction_reports - drug_reports)
D = len(all_report_ids - (drug_reports | reaction_reports))

print("A =", A)
print("B =", B)
print("C =", C)
print("D =", D)

A = 264
B = 1449
C = 254
D = 8033


In [107]:
# Calculate Reporting Odds Ratio (ROR) and its 95% confidence interval
def calculate_ror(A, B, C, D):

    if A == 0 or B == 0 or C == 0 or D == 0:
        return None

    ror = (A * D) / (B * C)

    se = math.sqrt(
        (1 / A) +
        (1 / B) +
        (1 / C) +
        (1 / D)
    )

    log_ror = math.log(ror)

    lower = math.exp(log_ror - 1.96 * se)
    upper = math.exp(log_ror + 1.96 * se)

    return ror, lower, upper

In [108]:
# Remove very rare drug-reaction pairs before calculating signals
# At least 3 reports are required for further analysis
MIN_REPORTS = 3

candidate_pairs = pair_counts[
    pair_counts["report_count"] >= MIN_REPORTS
].copy()

print("Candidate pairs:", len(candidate_pairs))

Candidate pairs: 3712


In [109]:
# Calculate contingency counts and ROR for candidate pairs
import math

results = []

all_report_ids = set(
    report.get("safetyreportid")
    for report in all_reports
)

for _, row in candidate_pairs.iterrows():

    drug = row["drug"]
    reaction = row["reaction"]

    drug_reports = set(
        drugs_df.loc[
            drugs_df["drug"] == drug,
            "report_id"
        ]
    )

    reaction_reports = set(
        reactions_df.loc[
            reactions_df["reaction"] == reaction,
            "report_id"
        ]
    )

    A = len(drug_reports & reaction_reports)
    B = len(drug_reports - reaction_reports)
    C = len(reaction_reports - drug_reports)
    D = len(all_report_ids - (drug_reports | reaction_reports))

    if A > 0 and B > 0 and C > 0 and D > 0:

        ror = (A * D) / (B * C)

        se = math.sqrt(
            (1 / A) +
            (1 / B) +
            (1 / C) +
            (1 / D)
        )

        log_ror = math.log(ror)

        lower = math.exp(log_ror - 1.96 * se)
        upper = math.exp(log_ror + 1.96 * se)

        results.append({
            "drug": drug,
            "reaction": reaction,
            "A": A,
            "B": B,
            "C": C,
            "D": D,
            "ROR": ror,
            "CI_lower": lower,
            "CI_upper": upper
        })

ror_df = pd.DataFrame(results)

In [110]:
# Rank drug-reaction pairs from highest to lowest ROR
ror_df = ror_df.sort_values("ROR", ascending=False).reset_index(drop=True)

In [111]:
ror_df.head(20)

,drug,reaction,A,B,C,D,ROR,CI_lower,CI_upper
0,SOBRIL,GALACTORRHOEA,3,1,1,9995,29985.000000,1501.828875,598670.221518
1,STRATTERA,BIPOLAR I DISORDER,3,1,1,9995,29985.000000,1501.828875,598670.221518
2,LITHIONIT,BREAST ENLARGEMENT,3,1,1,9995,29985.000000,1501.828875,598670.221518
3,LITHIONIT,GALACTORRHOEA,3,1,1,9995,29985.000000,1501.828875,598670.221518
4,CYCLIZINE,CAPILLARY LEAK SYNDROME,3,1,1,9995,29985.000000,1501.828875,598670.221518
5,SOBRIL,BREAST ENLARGEMENT,3,1,1,9995,29985.000000,1501.828875,598670.221518
6,IMATINIB,CHRONIC MYELOID LEUKAEMIA,5,1,2,9992,24980.000000,1939.582881,321718.863411
7,SOBRIL,BLOOD PROLACTIN INCREASED,3,1,2,9994,14991.000000,1054.983424,213017.641624
8,LITHIONIT,BLOOD PROLACTIN INCREASED,3,1,2,9994,14991.000000,1054.983424,213017.641624
9,IMATINIB,CHRONIC GRAFT VERSUS HOST DISEASE,5,1,4,9990,12487.500000,1178.796664,132285.457712


In [112]:
# Define potential signals using ROR and the lower confidence interval
ror_df["signal"] = (
    (ror_df["ROR"] > 1) &
    (ror_df["CI_lower"] > 1)
) #95

In [113]:
signals_df = ror_df[ror_df["signal"]].copy()

print("Signals:", len(signals_df))

Signals: 2502


In [114]:
# Filter and sort the drug-reaction pairs that meet the signal criteria
signals_df = signals_df.sort_values(
    "ROR",
    ascending=False
).reset_index(drop=True)

In [115]:
print("Potential signals:", len(signals_df))
signals_df.head(20)

Potential signals: 2502


,drug,reaction,A,B,C,D,ROR,CI_lower,CI_upper,signal
0,SOBRIL,GALACTORRHOEA,3,1,1,9995,29985.000000,1501.828875,598670.221518,True
1,SOBRIL,BREAST ENLARGEMENT,3,1,1,9995,29985.000000,1501.828875,598670.221518,True
2,STRATTERA,BIPOLAR I DISORDER,3,1,1,9995,29985.000000,1501.828875,598670.221518,True
3,LITHIONIT,GALACTORRHOEA,3,1,1,9995,29985.000000,1501.828875,598670.221518,True
4,CYCLIZINE,CAPILLARY LEAK SYNDROME,3,1,1,9995,29985.000000,1501.828875,598670.221518,True
5,LITHIONIT,BREAST ENLARGEMENT,3,1,1,9995,29985.000000,1501.828875,598670.221518,True
6,IMATINIB,CHRONIC MYELOID LEUKAEMIA,5,1,2,9992,24980.000000,1939.582881,321718.863411,True
7,LITHIONIT,BLOOD PROLACTIN INCREASED,3,1,2,9994,14991.000000,1054.983424,213017.641624,True
8,SOBRIL,BLOOD PROLACTIN INCREASED,3,1,2,9994,14991.000000,1054.983424,213017.641624,True
9,IMATINIB,CHRONIC GRAFT VERSUS HOST DISEASE,5,1,4,9990,12487.500000,1178.796664,132285.457712,True


In [116]:
signal_summary = signals_df[
    ["drug", "reaction", "A", "ROR", "CI_lower", "CI_upper"]
].copy()

signal_summary.head(20)

,drug,reaction,A,ROR,CI_lower,CI_upper
0,SOBRIL,GALACTORRHOEA,3,29985.000000,1501.828875,598670.221518
1,SOBRIL,BREAST ENLARGEMENT,3,29985.000000,1501.828875,598670.221518
2,STRATTERA,BIPOLAR I DISORDER,3,29985.000000,1501.828875,598670.221518
3,LITHIONIT,GALACTORRHOEA,3,29985.000000,1501.828875,598670.221518
4,CYCLIZINE,CAPILLARY LEAK SYNDROME,3,29985.000000,1501.828875,598670.221518
5,LITHIONIT,BREAST ENLARGEMENT,3,29985.000000,1501.828875,598670.221518
6,IMATINIB,CHRONIC MYELOID LEUKAEMIA,5,24980.000000,1939.582881,321718.863411
7,LITHIONIT,BLOOD PROLACTIN INCREASED,3,14991.000000,1054.983424,213017.641624
8,SOBRIL,BLOOD PROLACTIN INCREASED,3,14991.000000,1054.983424,213017.641624
9,IMATINIB,CHRONIC GRAFT VERSUS HOST DISEASE,5,12487.500000,1178.796664,132285.457712


In [117]:
signals_df["A"].describe()

,A
count,2502.000000
mean,6.673461
std,15.210324
min,3.000000
25%,3.000000
50%,4.000000
75%,6.000000
max,477.000000


In [118]:
signal_summary["report_count"] = signal_summary["A"]

## Preparing Results for Tableau

The final signal results are formatted into a simple table for visualization in Tableau.

In [119]:
# Select the columns needed for the Tableau dashboard
tableau_df = signal_summary[
    [
        "drug",
        "reaction",
        "report_count",
        "ROR",
        "CI_lower",
        "CI_upper"
    ]
].copy()

In [120]:
# Round statistical values to make the exported dataset easier to read
tableau_df["ROR"] = tableau_df["ROR"].round(2)
tableau_df["CI_lower"] = tableau_df["CI_lower"].round(2)
tableau_df["CI_upper"] = tableau_df["CI_upper"].round(2)

In [121]:
# Create a combined label for each drug-reaction pair
tableau_df["drug_reaction"] = (
    tableau_df["drug"] + " + " + tableau_df["reaction"]
)

In [125]:
# Export the final signal dataset as a CSV for Tableau
tableau_df.to_csv("drug_safety_signals.csv", index=False)

print("Exported successfully!")

Exported successfully!


In [123]:
tableau_df.shape

(2502, 7)

In [124]:
tableau_df.columns.tolist()

['drug',
 'reaction',
 'report_count',
 'ROR',
 'CI_lower',
 'CI_upper',
 'drug_reaction']